# EDA sơ bộ — Glints & JobsGO

Notebook kiểm kê và phân tích dữ liệu Raw Snapshot trong MinIO. Chạy sau khi `discover` và `detail` đã lưu dữ liệu thật. Notebook không sinh số liệu mẫu/giả.

**Output chính:** số lượng snapshot mới nhất theo tin, Missing Rate, tỷ lệ công khai lương dạng số, outlier IQR và biểu đồ phân phối. Lương chỉ được so sánh sau khi xác định được tiền tệ/chu kỳ trả lương; các trường raw không bị thay đổi.

In [ ]:
import os, re, json, math, warnings
from datetime import datetime, timezone
from collections import Counter
from pathlib import Path

import boto3
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'http://localhost:9000')
MINIO_BUCKET = os.getenv('MINIO_BUCKET', 'vn-it-jobs')
MINIO_ACCESS = os.getenv('MINIO_ROOT_USER', 'minioadmin')
MINIO_SECRET = os.getenv('MINIO_ROOT_PASSWORD', 'minioadmin')
MINIO_SECURE = os.getenv('MINIO_SECURE', 'false').lower() in {'1', 'true', 'yes'}
SOURCES = ['glints', 'jobsgo']
FX_VND_PER_USD = 25400  # chỉ dùng nếu nhóm thống nhất tỷ giá này cho kỳ phân tích

s3 = boto3.client(
    's3', endpoint_url=MINIO_ENDPOINT, aws_access_key_id=MINIO_ACCESS,
    aws_secret_access_key=MINIO_SECRET, use_ssl=MINIO_SECURE, region_name='us-east-1'
)
s3.head_bucket(Bucket=MINIO_BUCKET)
print('MinIO connected:', MINIO_ENDPOINT, '| bucket:', MINIO_BUCKET)

In [ ]:
def list_keys(prefix):
    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=MINIO_BUCKET, Prefix=prefix):
        keys.extend(item['Key'] for item in page.get('Contents', []) or [])
    return keys

def get_json(key):
    body = s3.get_object(Bucket=MINIO_BUCKET, Key=key)['Body'].read()
    return json.loads(body.decode('utf-8'))

def collect_latest_snapshots(source):
    # Only detail snapshots shaped <date>/<job_id>_vN.json; listing objects are excluded.
    pattern = re.compile(rf'^raw/{re.escape(source)}/\d{{4}}-\d{{2}}-\d{{2}}/.+_v(\d+)\.json$')
    latest = {}
    errors = []
    for key in list_keys(f'raw/{source}/'):
        match = pattern.match(key)
        if not match:
            continue
        try:
            obj = get_json(key)
            meta = obj.get('_meta', {})
            job_id = str(meta.get('source_job_id', ''))
            version = int(meta.get('version', match.group(1)))
            identity = job_id or key
            if identity not in latest or version > latest[identity][0]:
                latest[identity] = (version, key, obj)
        except Exception as exc:
            errors.append((key, str(exc)))
    rows = []
    for version, key, obj in latest.values():
        rows.append({'source': source, 'snapshot_key': key, 'version': version, 'snapshot': obj})
    return rows, errors

raw_snapshots = []
read_errors = []
for source in SOURCES:
    rows, errors = collect_latest_snapshots(source)
    raw_snapshots.extend(rows)
    read_errors.extend(errors)
print('Latest detail snapshots:', len(raw_snapshots))
print('Unreadable snapshots:', len(read_errors))
if not raw_snapshots:
    print('Chưa có detail snapshot để phân tích. Hãy chạy Phase 1 + Phase 2 cho Glints và JobsGO trước.')

In [ ]:
def all_strings(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for x in value.values():
            yield from all_strings(x)
    elif isinstance(value, (list, tuple)):
        for x in value:
            yield from all_strings(x)

def as_float(value):
    try:
        if value is None or isinstance(value, bool): return None
        x = float(str(value).replace(',', '').strip())
        return x if math.isfinite(x) else None
    except Exception:
        return None

def find_jsonld(html):
    soup = BeautifulSoup(html or '', 'html.parser')
    for script in soup.select('script[type="application/ld+json"]'):
        try: data = json.loads(script.string or script.get_text())
        except Exception: continue
        candidates = data if isinstance(data, list) else [data]
        for candidate in candidates:
            if not isinstance(candidate, dict): continue
            kind = candidate.get('@type')
            if kind == 'JobPosting' or (isinstance(kind, list) and 'JobPosting' in kind): return candidate
            for obj in candidate.get('@graph', []) if isinstance(candidate.get('@graph'), list) else []:
                if not isinstance(obj, dict): continue
                kind = obj.get('@type')
                if kind == 'JobPosting' or (isinstance(kind, list) and 'JobPosting' in kind): return obj
    return {}

def _parse_local_num(value, million_unit=False):
    text = str(value).strip().replace(' ', '')
    if million_unit:
        # In Vietnamese salary labels, comma is usually a decimal separator (20,5 triệu).
        return float(text.replace(',', '.'))
    # Native currency with explicit VND/USD marker: separators are thousands separators.
    return float(re.sub(r'[,.]', '', text))

def parse_salary(source, html, listing):
    """Extract a defensible salary value from raw metadata/text without changing Raw Zone."""
    salary_min = salary_max = None
    currency = None
    period = None
    salary_text = None
    value_unit = 'currency_native'
    monthly_markers = r'(?:/\s*tháng|mỗi\s*tháng|per\s*month|/\s*month|monthly|\btháng\b)'
    negotiable_re = re.compile(r'(?:thỏa\s*thuận|thoả\s*thuận|cạnh\s*tranh|negotiable|competitive)', re.I)

    if source == 'glints':
        candidates = listing.get('salaries') or []
        if candidates and isinstance(candidates[0], dict):
            sal = candidates[0]
            salary_min = as_float(sal.get('minAmount'))
            salary_max = as_float(sal.get('maxAmount'))
            currency = sal.get('CurrencyCode')
            period = sal.get('salaryMode')
        posting = find_jsonld(html)
        base = posting.get('baseSalary') or {}
        value = base.get('value') or {}
        if not isinstance(value, dict): value = {'value': value}
        if salary_min is None: salary_min = as_float(value.get('minValue'))
        if salary_max is None: salary_max = as_float(value.get('maxValue'))
        currency = base.get('currency') or currency
        period = value.get('unitText') or period
        if salary_min is not None or salary_max is not None:
            left = str(salary_min) if salary_min is not None else ''
            right = str(salary_max) if salary_max is not None else ''
            salary_text = (f'{left} - {right}'.strip(' -') + f' {currency or ""} / {period or "unknown period"}').strip()
    else:
        card_text = str(listing.get('card_text') or '')
        visible = BeautifulSoup(html or '', 'html.parser').get_text(' ', strip=True)
        candidates = [x for x in (card_text, visible) if x]
        # Do not use the entire card as a proxy for a non-null salary: extract a salary label/value.
        for text in candidates:
            negotiation = negotiable_re.search(text)
            if negotiation and re.search(r'(?:mức\s*lương|lương|salary)', text, re.I):
                salary_text = negotiation.group(0)
                break

        million_range = re.compile(r'(?P<min>\d+(?:[.,]\d+)?)\s*(?:-|–|đến|to)\s*(?P<max>\d+(?:[.,]\d+)?)\s*(?P<unit>triệu|tr\b)', re.I)
        million_single = re.compile(r'(?P<amount>\d+(?:[.,]\d+)?)\s*(?P<unit>triệu|tr\b)', re.I)
        native_range = re.compile(r'(?P<min>\d[\d.,]*)\s*(?:-|–|đến|to)\s*(?P<max>\d[\d.,]*)\s*(?P<currency>VNĐ|VND|USD|đ|\$)', re.I)
        native_single = re.compile(r'(?P<amount>\d[\d.,]*)\s*(?P<currency>VNĐ|VND|USD|đ|\$)', re.I)
        for text in candidates:
            m = million_range.search(text)
            if m:
                salary_min = _parse_local_num(m.group('min'), million_unit=True)
                salary_max = _parse_local_num(m.group('max'), million_unit=True)
                salary_text = m.group(0)
                currency, value_unit = 'VND', 'million'
                break
            m = re.search(r'(?:từ|from)\s*(?P<amount>\d+(?:[.,]\d+)?)\s*(?P<unit>triệu|tr\b)', text, re.I)
            if m:
                salary_min = _parse_local_num(m.group('amount'), million_unit=True)
                salary_text = m.group(0)
                currency, value_unit = 'VND', 'million'
                break
            m = re.search(r'(?:đến|tối đa|up\s*to)\s*(?P<amount>\d+(?:[.,]\d+)?)\s*(?P<unit>triệu|tr\b)', text, re.I)
            if m:
                salary_max = _parse_local_num(m.group('amount'), million_unit=True)
                salary_text = m.group(0)
                currency, value_unit = 'VND', 'million'
                break
            m = million_single.search(text)
            if m:
                salary_min = _parse_local_num(m.group('amount'), million_unit=True)
                salary_text = m.group(0)
                currency, value_unit = 'VND', 'million'
                break
            m = native_range.search(text)
            if m:
                salary_min = _parse_local_num(m.group('min'))
                salary_max = _parse_local_num(m.group('max'))
                salary_text = m.group(0)
                currency = 'USD' if m.group('currency').upper() in {'USD', '$'} else 'VND'
                break
            m = native_single.search(text)
            if m:
                salary_min = _parse_local_num(m.group('amount'))
                salary_text = m.group(0)
                currency = 'USD' if m.group('currency').upper() in {'USD', '$'} else 'VND'
                break
        all_visible = ' '.join(candidates)
        if re.search(monthly_markers, all_visible, re.I):
            period = 'month'

    # Use JSON-LD as an additional source only when the primary extraction lacked a value.
    if source == 'jobsgo' and salary_min is None and salary_max is None:
        posting = find_jsonld(html)
        base = posting.get('baseSalary') or {}
        value = base.get('value') or {}
        if not isinstance(value, dict): value = {'value': value}
        salary_min, salary_max = as_float(value.get('minValue')), as_float(value.get('maxValue'))
        currency = base.get('currency') or currency
        period = value.get('unitText') or period
        if salary_min is not None or salary_max is not None:
            salary_text = f'{salary_min if salary_min is not None else ""} - {salary_max if salary_max is not None else ""} {currency or ""} / {period or "unknown period"}'.strip()

    # Only values with an identified monthly period are comparable across listings.
    period_norm = str(period or '').casefold()
    monthly = period_norm in {'month', 'monthly', 'month(s)', 'per month'} or 'month' in period_norm or period_norm == 'tháng'
    if source == 'glints':
        if monthly and currency and str(currency).upper() == 'VND':
            min_m = salary_min / 1_000_000 if salary_min is not None else None
            max_m = salary_max / 1_000_000 if salary_max is not None else None
        elif monthly and currency and str(currency).upper() == 'USD':
            min_m = salary_min * FX_VND_PER_USD / 1_000_000 if salary_min is not None else None
            max_m = salary_max * FX_VND_PER_USD / 1_000_000 if salary_max is not None else None
        else:
            min_m = max_m = None
    else:
        min_m = max_m = None
        if monthly and currency and str(currency).upper() == 'VND':
            multiplier = 1.0 if value_unit == 'million' else 1.0 / 1_000_000
            min_m = salary_min * multiplier if salary_min is not None else None
            max_m = salary_max * multiplier if salary_max is not None else None
        elif monthly and currency and str(currency).upper() == 'USD':
            multiplier = FX_VND_PER_USD / 1_000_000
            min_m = salary_min * multiplier if salary_min is not None else None
            max_m = salary_max * multiplier if salary_max is not None else None

    numeric_public = salary_min is not None or salary_max is not None
    if salary_text is None and negotiable_re.search(' '.join(str(x) for x in (listing.get('card_text'), BeautifulSoup(html or '', 'html.parser').get_text(' ', strip=True)) if x)):
        salary_text = negotiable_re.search(' '.join(str(x) for x in (listing.get('card_text'), BeautifulSoup(html or '', 'html.parser').get_text(' ', strip=True)) if x)).group(0)
    return salary_text, salary_min, salary_max, currency, period, value_unit, min_m, max_m, numeric_public

def extract_record(row):
    snapshot = row['snapshot']
    meta = snapshot.get('_meta', {})
    raw = snapshot.get('raw', {})
    html = raw.get('html') or ''
    listing_wrap = raw.get('listing_payload') or {}
    listing = listing_wrap if isinstance(listing_wrap, dict) else {}
    soup = BeautifulSoup(html, 'html.parser')
    posting = find_jsonld(html)
    title = listing.get('title') or posting.get('title') or (soup.find('h1').get_text(' ', strip=True) if soup.find('h1') else None)
    company_obj = listing.get('company') or {}
    company = listing.get('company') if isinstance(listing.get('company'), str) else None
    company = company or company_obj.get('brandName') or company_obj.get('name')
    company = company or (posting.get('hiringOrganization') or {}).get('name')
    if not company and row['source'] == 'jobsgo':
        for anchor in soup.select('a[href]'):
            href = str(anchor.get('href') or '').lower()
            anchor_text = anchor.get_text(' ', strip=True)
            if anchor_text and any(token in href for token in ('cong-ty', '/company', 'nha-tuyen-dung')):
                company = anchor_text
                break
    visible = soup.get_text(' ', strip=True)
    salary_text, smin, smax, currency, period, salary_input_unit, smin_m, smax_m, numeric_public = parse_salary(row['source'], html, listing)
    if row['source'] == 'glints':
        loc_obj = listing.get('location') or {}
        location = loc_obj.get('formattedName') or loc_obj.get('name')
        experience = listing.get('minYearsOfExperience')
        if experience is None:
            exp = (posting.get('experienceRequirements') or {}).get('monthsOfExperience')
            experience = (exp / 12) if isinstance(exp, (int, float)) else None
        skill_vals = []
        for item in listing.get('skills') or []:
            if isinstance(item, dict) and isinstance(item.get('skill'), dict) and item['skill'].get('name'):
                skill_vals.append(str(item['skill']['name']))
        description = posting.get('description') or ''
    else:
        card = str(listing.get('card_text') or '')
        location = None
        # Prefer a DOM element with an explicit location label when present.
        for element in soup.find_all(['p', 'li', 'dd', 'div', 'span']):
            line = element.get_text(' ', strip=True)
            match = re.match(r'(?:địa điểm|nơi làm việc|location|work location)\s*:?\s*(.+)$', line, re.I)
            if match:
                candidate = match.group(1).strip()
                if candidate and len(candidate) <= 160:
                    location = candidate
                    break
        if not location:
            # Listing cards sometimes use pipe-delimited metadata instead of labels.
            skip_part = re.compile(r'(?:mức lương|salary|lương|triệu|\btr\b|vnđ|\bvnd\b|\busd\b|thỏa thuận|thoả thuận|cạnh tranh|kinh nghiệm|experience|\bnăm\b|\byears?\b|bằng cấp|hạn nộp|full[ -]?time|part[ -]?time|internship)', re.I)
            card_parts = [part.strip() for part in re.split(r'[|\n]+', card) if part.strip()]
            for candidate in card_parts:
                if candidate.casefold() == str(title or '').casefold() or skip_part.search(candidate):
                    continue
                if len(candidate) <= 100:
                    location = candidate
                    break
        if not location:
            m = re.search(r'(?:địa điểm|nơi làm việc|location)\s*:?\s*(.+?)(?=\s+(?:kinh nghiệm|hạn nộp|bằng cấp|thời gian làm việc|loại hình|cấp bậc|mô tả công việc|yêu cầu công việc)\s*:?|$)', visible, re.I)
            location = m.group(1).strip() if m else None
        if re.search(r'kinh nghiệm\s*:?\s*(?:không yêu cầu|không cần|no experience required)', visible, re.I):
            experience = 0.0
        else:
            m = re.search(r'kinh nghiệm\s*:?\s*(\d+(?:[.,]\d+)?)\s*(?:-|đến)?\s*(\d+(?:[.,]\d+)?)?\s*năm', visible, re.I)
            experience = float(m.group(1).replace(',', '.')) if m else None
        skill_vals = []
        description = visible
    # A transparent keyword scan is a derived feature, not a claim that JobsGO publishes raw skill tags.
    skill_terms = ['Python','Java','JavaScript','TypeScript','C#','.NET','C++','Go','PHP','SQL','MySQL','PostgreSQL','React','Vue.js','Angular','Node.js','Django','Flask','FastAPI','Docker','Kubernetes','AWS','Azure','GCP','Linux','Git','CI/CD','TensorFlow','PyTorch','Machine Learning','Data Engineering','Cybersecurity','Agile','Scrum']
    searchable = (str(description) + ' ' + visible).casefold()
    for term in skill_terms:
        if term.casefold() in searchable:
            skill_vals.append(term)
    # deduplicate skills preserving order
    skill_vals = list(dict.fromkeys(s for s in skill_vals if s))
    return {
        'source': row['source'], 'snapshot_key': row['snapshot_key'],
        'job_id': meta.get('source_job_id'), 'version': row['version'],
        'title': title, 'company': company, 'salary_text': salary_text or None,
        'salary_min_raw': smin, 'salary_max_raw': smax, 'salary_currency': currency,
        'salary_input_unit': salary_input_unit, 'salary_period': period, 'salary_min_million_vnd_month': smin_m,
        'salary_max_million_vnd_month': smax_m, 'salary_public_numeric': bool(numeric_public),
        'salary_negotiable': bool(salary_text and re.search(r'(?:thỏa\s*thuận|thoả\s*thuận|cạnh tranh|negotiable|competitive)', str(salary_text), re.I)),
        'salary_missing': bool(not salary_text and not numeric_public),
        'salary_comparable_monthly': bool(smin_m is not None or smax_m is not None),
        'location': location, 'min_experience_years': experience,
        'description_chars': len(BeautifulSoup(str(description), 'html.parser').get_text(' ', strip=True)),
        'skill_tags_raw': (listing.get('skills') if row['source'] == 'glints' else None),
        'skills_keyword_scan': skill_vals, 'html_chars': len(html),
    }

records = [extract_record(row) for row in raw_snapshots]
df = pd.DataFrame(records)
if not df.empty:
    display(df.head())
    print('Rows:', len(df), '| Sources:', df['source'].value_counts().to_dict())
else:
    print('DataFrame trống vì chưa có detail snapshots.')

In [ ]:
# Field Inventory: missing/null rates measured on latest snapshots, not fabricated examples.
if not df.empty:
    key_fields = ['title','company','salary_text','location','min_experience_years','description_chars','skill_tags_raw']
    inventory_rows = []
    for source, part in df.groupby('source'):
        for field in key_fields:
            values = part[field] if field in part.columns else pd.Series([None] * len(part), index=part.index)
            missing = values.isna()
            if field == 'skill_tags_raw':
                missing = values.apply(lambda x: x is None or (isinstance(x, (list, dict)) and len(x)==0))
            if field == 'description_chars': missing = values.fillna(0).eq(0)
            inventory_rows.append({'source': source, 'field': field, 'record_count': len(part),
                                   'null_count': int(missing.sum()), 'null_rate_pct': round(100*missing.mean(), 2)})
    missing_rate_table = pd.DataFrame(inventory_rows).sort_values(['source','null_rate_pct'], ascending=[True,False])
    display(missing_rate_table)
    salary_summary = df.groupby('source').agg(
        records=('job_id','count'),
        salary_numeric_count=('salary_public_numeric','sum'),
        salary_numeric_pct=('salary_public_numeric','mean'),
        comparable_monthly_count=('salary_comparable_monthly','sum'),
        comparable_monthly_pct=('salary_comparable_monthly','mean'),
        negotiable_salary_count=('salary_negotiable','sum'),
        negotiable_salary_pct=('salary_negotiable','mean'),
        missing_salary_count=('salary_missing','sum'),
        missing_salary_pct=('salary_missing','mean'),
    ).reset_index()
    for col in ['salary_numeric_pct','comparable_monthly_pct','negotiable_salary_pct','missing_salary_pct']:
        salary_summary[col] = (salary_summary[col] * 100).round(2)
    display(salary_summary)
else:
    missing_rate_table = pd.DataFrame()
    salary_summary = pd.DataFrame()

In [ ]:
# IQR outlier analysis for explicitly comparable monthly million-VND salary data.
def iqr_stats(series):
    x = pd.to_numeric(series, errors='coerce').dropna()
    if x.empty:
        return {'n':0,'min':None,'q1':None,'median':None,'q3':None,'max':None,'iqr':None,'lower':None,'upper':None,'outliers':0,'outlier_pct':None}
    q1, med, q3 = x.quantile(.25), x.median(), x.quantile(.75)
    iqr = q3 - q1
    lower, upper = max(0, q1 - 1.5*iqr), q3 + 1.5*iqr
    outliers = ((x < lower) | (x > upper))
    return {'n':len(x),'min':x.min(),'q1':q1,'median':med,'q3':q3,'max':x.max(),'iqr':iqr,
            'lower':lower,'upper':upper,'outliers':int(outliers.sum()),'outlier_pct':round(float(outliers.mean()*100),2)}

iqr_rows = []
if not df.empty:
    for source, part in df.groupby('source'):
        for column in ['salary_min_million_vnd_month','salary_max_million_vnd_month']:
            row = {'source':source, 'field':column, **iqr_stats(part[column])}
            iqr_rows.append(row)
iqr_summary = pd.DataFrame(iqr_rows)
display(iqr_summary) if not iqr_summary.empty else print('Chưa đủ dữ liệu lương số theo tháng để tính IQR.')

In [ ]:
# Distribution plots — separate figures, default matplotlib colors.
if not df.empty:
    numeric_salary = df[['source','salary_min_million_vnd_month','salary_max_million_vnd_month']].copy()
    for column in ['salary_min_million_vnd_month','salary_max_million_vnd_month']:
        data = pd.to_numeric(df[column], errors='coerce').dropna()
        if not data.empty:
            plt.figure(figsize=(8,4))
            plt.hist(data, bins=20)
            plt.title(f'Distribution: {column}')
            plt.xlabel('Million VND / month')
            plt.ylabel('Job count')
            plt.tight_layout()
            plt.show()
    for column in ['salary_min_million_vnd_month','salary_max_million_vnd_month']:
        grouped = [part[column].dropna() for _, part in df.groupby('source') if part[column].notna().any()]
        labels = [source for source, part in df.groupby('source') if part[column].notna().any()]
        if grouped:
            plt.figure(figsize=(7,4))
            plt.boxplot(grouped, labels=labels)
            plt.title(f'IQR check: {column}')
            plt.ylabel('Million VND / month')
            plt.tight_layout()
            plt.show()
else:
    print('Không có dữ liệu để vẽ.')

In [ ]:
# Top 15 keyword scan from raw detail text. For JobsGO this is derived NLP-lite, not a source tag field.
if not df.empty:
    counter = Counter()
    for items in df['skills_keyword_scan'].dropna():
        counter.update(set(items))
    if counter:
        top = pd.DataFrame(counter.most_common(15), columns=['skill_keyword','job_count'])
        display(top)
        plt.figure(figsize=(9,6))
        plt.barh(top['skill_keyword'][::-1], top['job_count'][::-1])
        plt.title('Top 15 technical skill keywords across latest snapshots')
        plt.xlabel('Jobs containing keyword')
        plt.tight_layout()
        plt.show()
    else:
        print('Chưa phát hiện từ khóa trong HTML snapshot.')
else:
    top = pd.DataFrame()

In [ ]:
# Source comparison: numeric salary disclosure rate. Negotiable/unknown salary remains distinct from zero.
if not df.empty:
    source_comparison = df.groupby('source').agg(
        total_latest_jobs=('job_id','count'),
        numeric_salary_jobs=('salary_public_numeric','sum'),
        negotiable_salary_jobs=('salary_negotiable','sum'),
        missing_salary_jobs=('salary_missing','sum'),
        missing_location_jobs=('location', lambda s: s.isna().sum()),
        missing_experience_jobs=('min_experience_years', lambda s: s.isna().sum()),
    ).reset_index()
    source_comparison['numeric_salary_pct'] = (source_comparison['numeric_salary_jobs']/source_comparison['total_latest_jobs']*100).round(2)
    source_comparison['negotiable_salary_pct'] = (source_comparison['negotiable_salary_jobs']/source_comparison['total_latest_jobs']*100).round(2)
    source_comparison['missing_salary_pct'] = (source_comparison['missing_salary_jobs']/source_comparison['total_latest_jobs']*100).round(2)
    source_comparison['missing_location_pct'] = (source_comparison['missing_location_jobs']/source_comparison['total_latest_jobs']*100).round(2)
    source_comparison['missing_experience_pct'] = (source_comparison['missing_experience_jobs']/source_comparison['total_latest_jobs']*100).round(2)
    display(source_comparison)
    plt.figure(figsize=(7,4))
    plt.bar(source_comparison['source'], source_comparison['numeric_salary_pct'])
    plt.title('Numeric salary disclosure rate by source')
    plt.ylabel('Percent of latest job snapshots')
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    source_comparison = pd.DataFrame()

# Optional export used as an evidence artifact for the report. It contains metrics only, no fabricated values.
if not df.empty:
    out_dir = Path('reports/generated')
    out_dir.mkdir(parents=True, exist_ok=True)
    run_date = datetime.now().strftime('%Y-%m-%d')
    df.drop(columns=['skill_tags_raw','skills_keyword_scan'], errors='ignore').to_csv(out_dir / f'glints_jobsgo_eda_rows_{run_date}.csv', index=False)
    missing_rate_table.to_csv(out_dir / f'glints_jobsgo_missing_rate_{run_date}.csv', index=False)
    iqr_summary.to_csv(out_dir / f'glints_jobsgo_iqr_{run_date}.csv', index=False)
    print('EDA evidence tables written under:', out_dir)

In [ ]:
# Raw Field Inventory: recursively count GraphQL/listing JSON paths and HTML selectors.
# This is intentionally generated from the latest raw snapshots, not hand-filled percentages.
def flatten_paths(value, prefix=''):
    out = []
    if isinstance(value, dict):
        for key, child in value.items():
            path = f'{prefix}.{key}' if prefix else str(key)
            out.extend(flatten_paths(child, path))
    elif isinstance(value, list):
        if not value:
            out.append((f'{prefix}[]', value))
        else:
            for child in value:
                out.extend(flatten_paths(child, f'{prefix}[]'))
    else:
        out.append((prefix, value))
    return out

def html_selector_presence(html):
    soup = BeautifulSoup(html or '', 'html.parser')
    found = set()
    for tag in soup.find_all(True):
        classes = sorted(c for c in (tag.get('class') or []) if c)
        selector = tag.name + ''.join('.' + re.sub(r'[^A-Za-z0-9_-]', '-', c) for c in classes)
        found.add('html_selector:' + selector)
        for attr in tag.attrs:
            if attr in {'class', 'style'}:
                continue
            found.add('html_attribute:' + tag.name + '[' + str(attr) + ']')
    return found

inventory_rows = []
for source in SOURCES:
    source_rows = [r for r in raw_snapshots if r['source'] == source]
    denominator = len(source_rows)
    if not denominator:
        continue
    observed = {}
    examples = {}
    for row in source_rows:
        raw = (row['snapshot'].get('raw') or {})
        listing = raw.get('listing_payload') or {}
        present = set()
        for path, value in flatten_paths(listing):
            present.add('listing_field:' + path)
            if path not in examples and value not in (None, '', [], {}):
                examples[path] = str(value)[:160]
        present.update(html_selector_presence(raw.get('html') or ''))
        for field in present:
            observed[field] = observed.get(field, 0) + 1
    for field, count in observed.items():
        inventory_rows.append({
            'source': source,
            'field_or_selector': field,
            'records_checked': denominator,
            'records_present': count,
            'presence_rate_pct': round(count * 100 / denominator, 2),
            'null_or_absent_rate_pct': round((denominator - count) * 100 / denominator, 2),
            'example_value': examples.get(field.split('listing_field:', 1)[-1], '') if field.startswith('listing_field:') else '',
        })
raw_field_inventory = pd.DataFrame(inventory_rows)
if not raw_field_inventory.empty:
    raw_field_inventory = raw_field_inventory.sort_values(['source','presence_rate_pct','field_or_selector'], ascending=[True,False,True])
    display(raw_field_inventory.head(100))
    out_dir = Path('reports/generated')
    out_dir.mkdir(parents=True, exist_ok=True)
    run_date = datetime.now().strftime('%Y-%m-%d')
    raw_field_inventory.to_csv(out_dir / f'glints_jobsgo_raw_field_inventory_{run_date}.csv', index=False)
    print('Raw field/selector inventory saved to reports/generated/.')
else:
    print('No rows to inventory yet; complete the detail crawl first.')